In [2]:
import pandas as pd
import openai
from qdrant_client import QdrantClient


from qdrant_client.models import Distance, VectorParams, PointStruct

In [3]:
df_items = pd.read_json('../../data/meta_Electronics_recent_2022_2023_has_main_category_ratings_100_sample_1000.jsonl', lines=True)

In [4]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,AMAZON FASHION,"Bosttor Bluetooth Beanie Hat with Light, Headl...",4.5,2342,"[100% Acrylic, Elastic closure, Hand Wash Only...",[],26.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Beanie Hat with Bluetooth Headphon...,Bosttor,"[Electronics, Headphones, Earbuds & Accessorie...","{'Department': 'unisex-adult', 'Date First Ava...",B0BH41HYFZ,NaN,NaN,NaN
1,All Electronics,"Aceele USB and USB C to Ethernet Adapter, 3.3f...",4.2,148,[【USB or USB C to Ethernet Adapter】The Etherne...,[],14.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Aceele,"[Electronics, Computers & Accessories, Network...",{'Product Dimensions': '3.54 x 0.98 x 0.71 inc...,B0BKPB2YQ9,NaN,NaN,NaN
2,All Electronics,HDMI Switch 3 in 1 Out 4K UHD HDMI Switcher Sp...,4.2,3331,[📍3-Port HDMI Switch: This aluminum HDMI switc...,[],18.98,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Darren reviews VWRHAR 3-1 splitter...,VWRHar,"[Electronics, Home Audio, Home Audio Accessori...",{'Package Dimensions': '6.1 x 4.21 x 0.94 inch...,B09MM5QT3R,NaN,NaN,NaN
3,All Electronics,"Smart Watch,Ip67 Waterproof Bluetooth Smartwat...",3.4,125,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Burxoe,[],{'Package Dimensions': '3.15 x 3.07 x 2.52 inc...,B09Q5TNDHY,NaN,NaN,NaN
4,Cell Phones & Accessories,"(3 Pack) Cute Airpod Case for Airpods 2&1,3D D...",4.7,335,[【Compatible Airpods 1/2 】This case designed f...,[1],11.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'airpod cases', 'url': 'https://www...",UGUHY,"[Electronics, Headphones, Earbuds & Accessorie...",{'Package Dimensions': '7.44 x 4.61 x 1.57 inc...,B0BZJKX7MS,NaN,NaN,NaN


In [5]:
list(df_items['features'].items())[0]


(0,
 ['100% Acrylic',
  'Elastic closure',
  'Hand Wash Only',
  '【Upgraded Bluetooth Beanie】Bosttor upgraded Bluetooth wireless technology offer much stable and strong connection, support music and calling, easy and fast to pair with your devices. Maximum transmission distance up to 33 feet. Built-in Stereo Speakers & Mic, enhance your music listening experience.',
  '【Long Working Time Music Hat】Built with easy-access USB charging port, simply charge it via the included USD cable. It takes 1-2 hours to get full charged. The battery offers continuous working hours up to 10 hours. Perfect for camping, hiking, skiing, hunting, jogging, cycling, dog walking, auto repair, etc.',
  '【LED Headlight Light Up Your Way】Hands-free LED light is rechargeable and removable. You can remove it to charge by laptop, power bank, socket, car charger, etc. The 4 LED lights can light up to 30 feet away. It points directional light toward what your eyes are looking at. It has 3 brightness settings.',
  '【O

In [6]:
list(df_items['images'].items())[0]

(0,
 [{'thumb': 'https://m.media-amazon.com/images/I/51b7qcj8dZL._AC_SR38,50_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51b7qcj8dZL._AC_.jpg',
   'variant': 'MAIN',
   'hi_res': 'https://m.media-amazon.com/images/I/71G+JK9lg1L._AC_UL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/51JvxXFcBwL._AC_SR38,50_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51JvxXFcBwL._AC_.jpg',
   'variant': 'PT01',
   'hi_res': 'https://m.media-amazon.com/images/I/71vYog2F1LL._AC_UL1300_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/51sKkBrOCiL._AC_SR38,50_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51sKkBrOCiL._AC_.jpg',
   'variant': 'PT02',
   'hi_res': 'https://m.media-amazon.com/images/I/71GowojGFfL._AC_UL1300_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/51k6uDxkZpL._AC_SR38,50_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51k6uDxkZpL._AC_.jpg',
   'variant': 'PT03',
   'hi_res': 'https://m.media-amazon.com/images/I/71ODAwq

In [7]:
def preprocess_description(row):
    return f"{row['title']} {' '.join(row['features'])}"

In [8]:
def extract_first_large_image(row):
    return row['images'][0].get('large','')

In [9]:
df_items['processed_description'] = df_items.apply(preprocess_description, axis=1)
df_items['image'] = df_items.apply(extract_first_large_image, axis=1)

In [10]:
df_items[['processed_description', 'image']].head()

,processed_description,image
0,"Bosttor Bluetooth Beanie Hat with Light, Headl...",https://m.media-amazon.com/images/I/51b7qcj8dZ...
1,"Aceele USB and USB C to Ethernet Adapter, 3.3f...",https://m.media-amazon.com/images/I/41WsGRr-3T...
2,HDMI Switch 3 in 1 Out 4K UHD HDMI Switcher Sp...,https://m.media-amazon.com/images/I/41fsjaknf1...
3,"Smart Watch,Ip67 Waterproof Bluetooth Smartwat...",https://m.media-amazon.com/images/I/51YFypeuZh...
4,"(3 Pack) Cute Airpod Case for Airpods 2&1,3D D...",https://m.media-amazon.com/images/I/41V7Wi2ckb...


In [11]:
df_sample = df_items.sample(50, random_state=42)

In [12]:
df_sample.shape

(50, 18)

In [13]:
relevant_columns = ['processed_description', 'image', 'rating_number', 'price', 'average_rating','parent_asin']
df_data_to_embed = df_sample[relevant_columns]

In [14]:
df_data_to_embed.head()

,processed_description,image,rating_number,price,average_rating,parent_asin
521,Marame 120mm 5v USB Powered Fan with Speed Con...,https://m.media-amazon.com/images/I/41Zpg4vSEj...,104,14.99,4.7,B0BRJS644Z
737,"Kids Wireless Headphones, Adjustable Headband,...",https://m.media-amazon.com/images/I/41+jaEbUbY...,173,NaN,4.2,B09KQP2H7N
740,"Veetone Aux Cord for iPhone, 3.3ft [Apple MFi ...",https://m.media-amazon.com/images/I/31ag2Blvjm...,2218,5.99,4.0,B0CC4HBS85
660,AICHESON Laptop Cooler Pad with 6 Cooling Fans...,https://m.media-amazon.com/images/I/51BKtk3-RD...,145,NaN,4.2,B099N9F3FP
411,"Raymate Bluetooth Speakers, HiFi Stereo Sound ...",https://m.media-amazon.com/images/I/31SnPdXS77...,270,38.69,4.7,B0C996WY16


In [15]:
data_to_embed = df_data_to_embed.to_dict(orient='records')

In [16]:
data_to_embed[0]

{'processed_description': 'Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) 【Solve Your Cooling Problem】\xa0This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed\xa0Control\xa0Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility USB-Powered】 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that support USB connection. USB fan is energy-saving and environmentally friendly. 【Rubber Feet & Dust Filter】\xa0Rubber Feet on the fan raise it up enough off of the surface it is sitting on

In [18]:
from openai import OpenAI

client = OpenAI(
    api_key=config.GOOGLE_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

response = client.embeddings.create(
    model="gemini-embedding-001",
    input="Random text",
)

NameError: name 'config' is not defined

In [ ]:
len(response.data[0].embedding)

In [18]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

True

In [19]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

response = client.embeddings.create(
    model="gemini-embedding-001",
    input="Random text",
)

print(response.data[0].embedding)

[-0.014552347362041473, 0.03157435357570648, 0.012304088100790977, -0.04824463650584221, 0.0036319042555987835, 0.02266230620443821, 0.0013859155587852001, -0.008250858634710312, -0.008741911500692368, 0.0062147281132638454, 0.016333764418959618, -0.006102106999605894, -0.005134551785886288, 0.03893044963479042, 0.0960603877902031, 0.0037646652199327946, -0.014426860958337784, 0.028704147785902023, -0.0035096858628094196, -0.014929561875760555, -0.0007851755945011973, 0.0072447857819497585, 0.017399318516254425, -0.010492537170648575, -0.011349381878972054, -0.003866474609822035, 0.007899541407823563, -0.0012400973355397582, 0.025804730132222176, 0.008903557434678078, -0.0008346645045094192, 0.026523876935243607, -0.0012324982089921832, -0.0007423584465868771, 0.006199580617249012, 0.024468017742037773, -0.009936532936990261, 0.007072800304740667, -0.0033930903300642967, 0.035918090492486954, -0.022724542766809464, -0.015448431484401226, -0.0027485168538987637, -0.009622892364859581, 0

In [20]:
len(response.data[0].embedding)

3072

In [25]:
get_embedding('Random text')

AuthenticationError: Error code: 401 - Incorrect API key provided: your_goo*******_key. You can find your API key at https://platform.openai.com/account/api-keys.

In [25]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)


def get_embedding(text, model="gemini-embedding-001"):
    response = client.embeddings.create(
        model=model,
        input=text,
    )
    return response.data[0].embedding


vector = get_embedding("Random text")
print(len(vector))
print(vector[:5])

3072
[-0.014552347362041473, 0.03157435357570648, 0.012304088100790977, -0.04824463650584221, 0.0036319042555987835]


In [26]:

vector = get_embedding("Alexio")
print(len(vector))
print(vector[:5])

3072
[0.016062412410974503, -0.02674209140241146, -0.018382463604211807, -0.07514583319425583, -0.005315020214766264]


In [27]:
qdrant_client = QdrantClient(
    url='http://localhost:6333'
)

In [28]:
qdrant_client.create_collection(
    collection_name='amazon-items-collection-00',
    vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
)

UnexpectedResponse: Unexpected Response: 409 (Conflict)
Raw response content:
b'{"status":{"error":"Wrong input: Collection `amazon-items-collection-00` already exists!"},"time":0.000375119}'

In [29]:
from qdrant_client.models import PointStruct

point_struct = PointStruct(
    id=1,
    vector=get_embedding("Hello world"),
    payload={
        "text": "Hello world",
        "model": "gemini-embedding-001",
    },
)

point_struct

PointStruct(id=1, vector=[-0.023421520367264748, 0.016765719279646873, 0.009261323139071465, -0.06383000314235687, -0.00262627680785954, 0.0010187155567109585, -0.011256840080022812, 0.013003658503293991, 0.008751670829951763, 0.0012745214626193047, -0.000788087781984359, -0.019086141139268875, 0.030971918255090714, 0.04426491633057594, 0.11137618869543076, 0.01802559569478035, -0.0031383372843265533, -0.013070215471088886, 0.0054121012799441814, -0.009036983363330364, 0.009904046542942524, -0.01071805227547884, 0.012551899068057537, 0.011420718394219875, -0.03375867009162903, 0.00887706596404314, 0.0277958232909441, 0.0037452082615345716, 0.029841817915439606, 0.01986183598637581, 0.018075233325362206, -0.00717808585613966, -0.024979570880532265, 0.011720823124051094, -0.007332403678447008, 0.006851526442915201, 0.011618152260780334, 0.00427890382707119, -0.009942627511918545, -0.009696085937321186, -0.00370280840434134, 0.006349137052893639, -0.012098702602088451, -0.0156292710453271

### Amazon items

In [30]:
point_structs = []
for i, row in enumerate(data_to_embed):
    point_structs.append(
        PointStruct(
            id=i,
            vector=get_embedding(row['processed_description']),
            payload=row
        )
    )
    

In [31]:
point_structs


[PointStruct(id=0, vector=[0.017121920362114906, -0.029839590191841125, 0.026352619752287865, -0.06911855190992355, 0.015927337110042572, 0.019641395658254623, -0.011132761836051941, 0.010832282714545727, 0.0012275110930204391, 0.012119865976274014, -0.016949985176324844, -0.0396931953728199, -0.005422365851700306, 0.0022024528589099646, 0.10199838131666183, -0.013109077699482441, 0.004802503157407045, -0.010530938394367695, 0.028392454609274864, -0.007512554991990328, -0.011927185580134392, -0.021080153062939644, -0.016345642507076263, -0.008186152204871178, 0.002198397647589445, -0.002610014285892248, 0.011549077928066254, 0.015178149566054344, 0.006718732882291079, -0.006960943806916475, -0.026118231937289238, -0.005738997366279364, 0.02346726506948471, 0.015696728602051735, 0.016133449971675873, -0.005946400109678507, 0.02451745606958866, -0.017875291407108307, -0.009965452365577221, 0.017149759456515312, -0.035524994134902954, -0.0001752950920490548, 0.0007605661521665752, 0.01097

In [32]:
len(point_structs)

50

In [33]:
qdrant_client.upsert(
    collection_name='amazon-items-collection-00',
    points=point_structs
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [34]:
def retrieve_data(query, collection_name='amazon-items-collection-00', k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_embedding,
        limit=k
    )
    return results

In [35]:
retrieve_data('Can I get a recommendation for a new headphone?').points

[ScoredPoint(id=37, version=1, score=0.5908841, payload={'processed_description': "Sleep Headphones Bluet00th Headband Wireless - Sleeping Headphones Adjustable [FIT for Side Sleepers,Headband Headphones - Sleep Mask with Bluet00th Headphones - Sleep Noise Cancelling [Sleep Headphone for Side Sleeper and Sleep Aid] : Bluet00th headband is perfect solution for men women who is insomnia-early, the ultra-thin earbuds of sleep headband is nestled into an extremely stretchy fabric so you can't feel them even when your ear or head is pressed into a pillow. Minimalist line design and soft fabrics offer you a super brand and better sleeping experience. The headband can double up as workout earphones. Cool for sport running meditation Yoga airplane. [Unique Gifts Idea for Women Men Kids Teens] : Wireless Bluet00th headphone is a unique 3 in 1 headband mask: Sleep Headphones & Sleep Mask & Sports Headband. Elastic fabric with comfortable experience, suit for most women men adults. Bluet00th slee